# AIRPATH-AI Milestone 4 — route exposure aggregation validation

This notebook evaluates the offline route index

`E(route) = Σ segment_PM2.5 × segment_duration_minutes`.

The index is a time-weighted PM2.5 exposure proxy in `(µg/m³)·min`, not inhaled dose. Oracle and predicted pathways remain separate. Candidate ranks are evaluation labels only: this notebook does not optimize, recommend, or constrain routes, and it does not validate minute-level PM2.5.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.exposure import generate_exposure_outputs

In [ ]:
output_directory = PROJECT_ROOT / "data/processed/exposure"
outputs = generate_exposure_outputs(
    prediction_csv=PROJECT_ROOT / "data/processed/xgboost_forecasting_predictions.csv",
    network_path=PROJECT_ROOT / "data/processed/road_network/healthyair_pilot_osm.json.gz",
    output_directory=output_directory,
    report_path=PROJECT_ROOT / "reports/route_exposure.md",
)
outputs["route_summary"]

In [ ]:
display(outputs["error_metrics"])
display(outputs["agreement"])
display(outputs["ranking"])

In [ ]:
display(
    outputs["top_segments"][[
        "scenario_id",
        "mode",
        "route_id",
        "pipeline_mode",
        "segment_index",
        "pm25_estimate",
        "segment_duration_minutes",
        "exposure_contribution",
        "contribution_fraction",
    ]].head(20)
)
display(outputs["relationships"])

In [ ]:
outputs["decision"]

In [ ]:
for filename in (
    "candidate_exposure_error.png",
    "ranking_correlation.png",
    "ranking_agreement.png",
):
    display(Image(filename=output_directory / filename))

## Interpretation boundary

The analysis uses hourly target mapping and an oracle-IDW internal reference, not road measurements. Predicted exposure underestimation, limited OD coverage, highly overlapping alternatives, and one scenario's weaker rank agreement require a restricted decision.

No candidate is designated as recommended. Any later constrained-optimization milestone must remain offline until broader OD, temporal, and road-level validation exists.